In [0]:
from pyspark.sql.functions import *

# Exploratory Data Analysis silver
- Here im checking the data and changing the names of the columns 
- Because the new ones will be easier to use for coding

In [0]:
# Läser in data från bronze

df = spark.table("marathos.bronze.results_raw")

display(df)

In [0]:
df_clean = (
    df
    .withColumnRenamed("Year of event", "event_year")
    .withColumnRenamed("Event dates", "event_dates")
    .withColumnRenamed("Event name", "event_name")
    .withColumnRenamed("Event distance/length", "event_distance_length")
    .withColumnRenamed("Event number of finishers", "event_number_of_finishers")
    .withColumnRenamed("Athlete performance", "athlete_performance")
    .withColumnRenamed("Athlete club", "athlete_club")
    .withColumnRenamed("Athlete country", "athlete_country")
    .withColumnRenamed("Athlete year of birth", "athlete_year_of_birth")
    .withColumnRenamed("Athlete gender", "athlete_gender")
    .withColumnRenamed("Athlete age category", "athlete_age_category")
    .withColumnRenamed("Athlete average speed", "athlete_average_speed")
    .withColumnRenamed("Athlete ID", "athlete_id")
)

display(df_clean)

In [0]:
df_clean.printSchema()

### 

In [0]:
display(
    df_clean.select("event_distance_length")
    .distinct()
    .orderBy("event_distance_length")
)

In [0]:
display(
    df_clean.select("athlete_performance")
    .distinct()
)

### Removing all the d (days)
- Here i'm observing if there is any "d" in the athlete_perfomance
- The reason behind it is that it will look more neater

In [0]:
# Kollar igenom om det finns några event med "d" i athlete_performance kolumnen
display(
    df_clean.filter(
        col("athlete_performance").contains("d")
    )
)

In [0]:
# Filtrerar bort event med "d" i athlete_performance kolumnen
df_clean = df_clean.filter(
    ~col("athlete_performance").contains("d")
)

display(df_clean)

In [0]:
# Kollar igenom om det finns några event med "d" i athlete_performance kolumnen
display(
    df_clean.filter(
        col("athlete_performance").contains("d")
    )
)

### Cleaning the distance_length table
- Here i'm checking in the event_distance_length and event_type
- I've found some strange words in the event_distance_length which i wanted to remove
- I also wanted to clean that km and mi is in the distance category and h is in time, so it would look more organized

In [0]:
# Kollar igenom event_distance_length kolumnen och event_type kolumnen

df_clean = df_clean.withColumn(
    "event_type",
    when(col("event_distance_length").contains("km"), "distance")
    .when(col("event_distance_length").contains("mi"), "distance")
    .when(col("event_distance_length").contains("h"), "time")
    .otherwise("unknown")
)

display(
    df_clean.select(
        "event_distance_length",
        "event_type"
    ).distinct()
)

In [0]:
# Kollar igenom om det finns några event med "Etappen" i event_distance_length kolumnen
df_clean = df_clean.filter(
    ~col("event_distance_length").contains("Etappen")
)

display(df_clean)

In [0]:
# Kollar om alla "Etappen" eventen är bortaggna från event_distance_length kolumnen
display(
    df_clean.filter(
        col("event_distance_length").contains("Etappen")
    )
)

### Creating a new column
- I wanted to create a new column called event_id sp that everyone in the event_name column gets their own unique id
- It's better that they have their own unique id because it will be easier to find and there won't be any mix up

In [0]:
# Skapar en ny kolumn event_id så att alla event_name får en unik id
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

event_window = Window.orderBy("event_name")

df_clean = df_clean.withColumn(
    "event_id",
    dense_rank().over(event_window)
)

display(
    df_clean.select(
        "event_name",
        "event_id"
    ).distinct()
)

In [0]:
# Dubbelkollar att event_id är unika
display(
    df_clean.filter(
        col("event_name").contains("Selva Costera")
    ).select(
        "event_name",
        "event_id"
    )
)

In [0]:
# Skapar en ny kolumn result_id så att alla event_name, athlete_id och athlete_performance får en unik id
result_window = Window.orderBy(
    "event_name",
    "athlete_id",
    "athlete_performance"
)

df_clean = df_clean.withColumn(
    "result_id",
    dense_rank().over(result_window)
)

display(
    df_clean.select(
        "result_id",
        "event_name",
        "athlete_id",
        "athlete_performance"
    )
)

### Cleaning data in athlete_performance and event_type

In [0]:
# Kollar igenom event typen och athlete_performance kolumnen och rensar
display(
    df_clean.filter(
        (col("event_type") == "distance") &
        (~col("athlete_performance").contains("h"))
    )
)

In [0]:
# Fixar till athlete_club kolumnen genom att få bort nullvärden
# Dubbelkollar athlete_performance och event_type
df_clean = df_clean.filter(
    ~(
        (col("event_type") == "distance") &
        (~col("athlete_performance").contains("h"))
    )
)

df_clean = df_clean.withColumn(
    "athlete_club",
    when(col("athlete_club").isNull(), "Unknown")
    .otherwise(trim(col("athlete_club")))
)

display(df_clean)

### Removing the * from athlete_club

In [0]:
# Tar bort alla * tecken från athlete_club
df_clean = df_clean.withColumn(
    "athlete_club",
    regexp_replace(col("athlete_club"), r"\*", "")
)

In [0]:
# Visar athlete_club kolumnen och dubbelkollar att alla * har tagits bort
display(
    df_clean.select("athlete_club").distinct()
)

### Changing the average_speed to two decimals

In [0]:
# Avrundar athlete_average_speed kolumnen till två decimaler
# Konverterar datatypen till double
df_clean = df_clean.withColumn(
    "athlete_average_speed",
    expr("try_cast(athlete_average_speed as double)")
)

df_clean = df_clean.withColumn(
    "athlete_average_speed",
    round(col("athlete_average_speed"), 2)
)

display(df_clean)

### Changing the date format

In [0]:
# Ändrar om datumen till dd-mm-yyyy format
df_clean = df_clean.withColumn(
    "event_date_clean",
    regexp_extract(col("event_dates"), r"(\d{2}\.\d{2}\.\d{4})$", 1)
)

df_clean = df_clean.withColumn(
    "event_date",
    to_date(col("event_date_clean"), "dd.MM.yyyy")
)

display(
    df_clean.select(
        "event_dates",
        "event_date_clean",
        "event_date"
    )
)

In [0]:
.withColumn(
    "athlete_year_of_birth",
    col("athlete_year_of_birth").cast("int")
)

.withColumn(
    "athlete_age_category",
    when(
        col("athlete_age_category").isNull(),
        "Unknown"
    ).otherwise(trim(col("athlete_age_category")))
)

### Checking at the cleaned data

In [0]:
# Visar fram den städade datan
display(spark.table("marathos.silver.marathos_obt"))

In [0]:
spark.table("marathos.silver.marathos_obt").printSchema()